# 8 — Filtered training (impossible-label cleaning) — HGT + HAN

This notebook is the **data-cleaning experiment** decided with the professor
(recommendation #4). It is a *self-contained additive run*: it does **not**
change notebooks 3/5/6 or any baseline output, so the original heterogeneous /
homogeneous comparison stays reproducible.

**What it does differently from notebooks 5 & 6**

1. Applies a new opt-in transform, `ImpossibleLabelFilter`, to the loaded graphs
   **before** any other step. It removes every beam/column whose smaller side
   `min(b, h) <= 10 cm` — exactly the 449 physically-impossible labels
   (`5x5`, `10x10`, `30x10`). Every legitimate section (15, 20, 25 cm ...) is kept.
2. The filter is applied to **both** the training graphs **and** the external
   test graphs (`data/graphs/test_graphs.pt`), because the impossible labels are
   corrupt ground truth in both splits.
3. It trains **both models** (HGT and HAN) with the identical sweep / CV /
   deployment / external-test pipeline used in notebooks 5 & 6.
4. All outputs are written under **`results/filtered/<model>/`**, mirroring the
   existing `results/<model>/` layout, so the baseline results are never touched.

Nothing on disk is modified except the new `results/filtered/` tree — the `.pt`
graph files and the source Excel are read-only here.


In [1]:
# ## Cell 1 — Setup, imports, device
import os, sys, copy, json, itertools, warnings, logging
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch_geometric
from torch.serialization import add_safe_globals
warnings.filterwarnings("ignore")

sys.path.append("..")
sys.path.append("../src")

logging.basicConfig(level=logging.INFO,
    format="%(asctime)s | %(name)-8s | %(levelname)-8s | %(message)s",
    datefmt="%H:%M:%S")
logger = logging.getLogger("NB8")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.float_format", "{:.4f}".format)

def detect_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = detect_device()
print("PyTorch:", torch.__version__, "| PyG:", torch_geometric.__version__,
      "| device:", device)

# let torch.load rebuild HeteroData objects
add_safe_globals([
    torch_geometric.data.storage.BaseStorage,
    torch_geometric.data.storage.NodeStorage,
    torch_geometric.data.storage.EdgeStorage,
    torch_geometric.data.HeteroData,
])
print("Setup complete")


PyTorch: 2.5.1+cu121 | PyG: 2.8.0 | device: cuda
Setup complete


In [2]:
# ## Cell 2 — Config + filter settings
import yaml
from src.models import build_model
from src.training.trainer import Trainer
from src.data_manager.data_processor import (
    ImpossibleLabelFilter, IsolatedNodeHandler, PositionalEncoder,
    FeatureNormalizer, TargetNormalizer,
)

with open("../configs/base.yaml") as f:
    base_config = yaml.safe_load(f)

def load_model_config(model_type):
    """Merge base.yaml with the per-model yaml (hgt.yaml / han.yaml) exactly like
    notebooks 5 & 6, and force model.type so build_model() picks the right op."""
    path = f"../configs/models/{model_type}.yaml"
    with open(path) as f:
        mc = yaml.safe_load(f)
    cfg = {**base_config, **mc}
    cfg["model"]["type"] = model_type
    return cfg

# ---- experiment knobs -------------------------------------------------------
MODELS          = ["hgt", "han"]     # train BOTH
FILTER_MAX_CM   = 10.0               # remove nodes with min(b,h) <= this (the 449)
APPLY_TO_TEST   = True               # clean the external test set too
RESULTS_ROOT    = "../results/filtered"
TRAIN_PT        = "../data/graphs/train_graphs.pt"
TEST_PT         = "../data/graphs/test_graphs.pt"
HIT_TOL_CM      = 5.0                # "within X cm" hit-rate threshold
os.makedirs(RESULTS_ROOT, exist_ok=True)

# pe / isolated / cv settings are shared across models (read from base config)
KNN_K   = base_config["data"]["isolated"].get("knn_k", 4)
PE_DIM  = base_config["data"]["pe"].get("dim", 8)
N_FOLDS = max(2, min(base_config.get("training", {}).get("k_folds", 5), 5))
print(f"Filter: remove min(b,h) <= {FILTER_MAX_CM} cm | apply_to_test={APPLY_TO_TEST}")
print(f"Sweep : pe_dim={PE_DIM} | knn_k={KNN_K} | folds={N_FOLDS} | tol={HIT_TOL_CM}cm")
print(f"Output: {RESULTS_ROOT}/<model>/ (baseline results/<model>/ untouched)")


Filter: remove min(b,h) <= 10.0 cm | apply_to_test=True
Sweep : pe_dim=8 | knn_k=4 | folds=5 | tol=5.0cm
Output: ../results/filtered/<model>/ (baseline results/<model>/ untouched)


In [3]:
# ## Cell 3 — Load graphs + preview what the filter removes (sanity check)
def load_graphs(pt_path):
    """Return a plain list of HeteroData with sample_name attached."""
    loaded = torch.load(pt_path, weights_only=False)
    if isinstance(loaded, dict):
        loaded = list(loaded.values())
    if len(loaded) and isinstance(loaded[0], tuple):
        gs = []
        for nm, g in loaded:
            if not hasattr(g, "sample_name"):
                g.sample_name = nm
            gs.append(g)
        return gs
    return list(loaded)

def count_impossible(graphs, thr):
    """How many beam/column nodes have min(b,h) <= thr (per size)."""
    from collections import Counter
    c = Counter(); total = 0
    for g in graphs:
        for nt in ("beam", "column"):
            if nt not in g.node_types or not hasattr(g[nt], "y"):
                continue
            y = g[nt].y
            if y is None or y.numel() == 0:
                continue
            bad = (y.min(dim=1).values <= thr)
            total += y.shape[0]
            for i in torch.nonzero(bad).flatten().tolist():
                b, h = float(y[i, 0]), float(y[i, 1])
                c[f"{int(b)}x{int(h)}"] += 1
    return total, c

raw_train = load_graphs(TRAIN_PT)
raw_test  = load_graphs(TEST_PT)
print(f"Loaded {len(raw_train)} train graphs, {len(raw_test)} test graphs\n")

for split, gs in [("train", raw_train), ("test", raw_test)]:
    tot, c = count_impossible(gs, FILTER_MAX_CM)
    n_bad = sum(c.values())
    print(f"[{split}] {tot} nodes | impossible (min<= {FILTER_MAX_CM:.0f}cm): "
          f"{n_bad}  -> {dict(sorted(c.items()))}")
print("\n(These are what the filter will drop. Legitimate sizes are untouched.)")


Loaded 254 train graphs, 59 test graphs

[train] 56748 nodes | impossible (min<= 10cm): 251  -> {'10x10': 111, '5x5': 140}
[test] 13862 nodes | impossible (min<= 10cm): 198  -> {'10x10': 97, '30x10': 15, '5x5': 86}

(These are what the filter will drop. Legitimate sizes are untouched.)


In [4]:
# ## Cell 4 — Shared pipeline helpers (sweep + external test), identical
# ##          metric logic to notebooks 5 & 6, parametrised by model + output dir.

def make_model(config):
    return build_model(config)

def _reg(true, pred, tol):
    """MAE/RMSE/R2 + hit-rate from (N,2)=[width,height]; R2 per dim, uniform-avg."""
    keys = ["mae", "rmse", "width_mae", "height_mae", "width_rmse", "height_rmse",
            "width_r2", "height_r2", "r2",
            "within_tol", "width_within_tol", "height_within_tol"]
    if true.shape[0] == 0:
        return {k: float("nan") for k in keys}
    err = pred - true
    r = {"mae": float(np.abs(err).mean()),
         "rmse": float(np.sqrt((err ** 2).mean())),
         "within_tol": float((np.abs(err) <= tol).mean())}
    for di, nm in enumerate(["width", "height"]):
        e = err[:, di]; y = true[:, di]
        r[f"{nm}_mae"] = float(np.abs(e).mean())
        r[f"{nm}_rmse"] = float(np.sqrt((e ** 2).mean()))
        r[f"{nm}_within_tol"] = float((np.abs(e) <= tol).mean())
        ss_res = float((e ** 2).sum()); ss_tot = float(((y - y.mean()) ** 2).sum())
        r[f"{nm}_r2"] = (1 - ss_res / ss_tot) if ss_tot > 0 else float("nan")
    r["r2"] = float(np.nanmean([r["width_r2"], r["height_r2"]]))
    return r

def _at(mm, k, i):
    v = mm.get(k); return v[i] if (v and 0 <= i < len(v)) else float("nan")

def _g(m, k):
    return float(m.get(k, float("nan")))


def run_sweep(config, raw_graphs, results_dir, models_dir, hit_tol=HIT_TOL_CM):
    """Full (PE x isolated) sweep with k-fold CV + a held-out 15% internal test.
    `raw_graphs` must ALREADY be filtered. Returns (results_table, best_combo)."""
    os.makedirs(models_dir, exist_ok=True); os.makedirs(results_dir, exist_ok=True)
    nG = len(raw_graphs)
    has_names = hasattr(raw_graphs[0]["beam"], "feature_names")
    pe_modes = ["topological", "geometric", "hybrid"] if has_names else ["topological"]
    iso_strategies = ["none", "self_loop", "knn"]

    np.random.seed(42); idx = np.random.permutation(nG); cut = int(nG * 0.85)
    train_idx, test_idx = idx[:cut], idx[cut:]
    combos = list(itertools.product(pe_modes, iso_strategies))
    print(f"[{config['model']['type'].upper()}] sweeping {len(combos)} combos | "
          f"{N_FOLDS}-fold CV + final each (~{len(combos)*(N_FOLDS+1)} trainings).\n")

    sweep_rows, sweep_preds, breakdown_rows, persample_rows, pernode_rows = [], {}, [], [], []
    for ci, (pe_mode, iso) in enumerate(combos, 1):
        tag = f"{pe_mode}_{iso}"
        print("=" * 72)
        print(f"COMBO {ci}/{len(combos)} | PE={pe_mode} | isolated={iso}"
              + (f" (k={KNN_K})" if iso == "knn" else ""))
        print("=" * 72)
        try:
            g_all = [raw_graphs[i].clone() for i in range(nG)]
            IsolatedNodeHandler(strategy=iso, k=KNN_K).transform(g_all)
            PositionalEncoder(mode=pe_mode, dim=PE_DIM).transform(g_all)
            train_pool_s = [g_all[i] for i in train_idx]
            test_s       = [g_all[i] for i in test_idx]

            cfg = copy.deepcopy(config); cfg["paths"] = {"checkpoints": f"{models_dir}/{tag}"}

            tr = Trainer(model=make_model(cfg), config=cfg, show_progress=True)
            cv = tr.cross_validate(train_pool_s, n_folds=N_FOLDS, shuffle=True, random_state=42)
            print("\n  [CV] per-fold val (cm):")
            for fr in cv["fold_results"]:
                be = fr["best_epoch"] - 1; mm = fr["metrics"]
                print(f"    fold {fr['tag'].split('_')[-1]}: "
                      f"overall {_at(mm,'val_overall_mae',be):.3f} | "
                      f"beam {_at(mm,'val_beam_mae',be):.3f} | "
                      f"col {_at(mm,'val_column_mae',be):.3f}")
            print(f"  [CV] mean overall MAE = {cv.get('mean_best_val_overall_mae', float('nan')):.3f}"
                  f" +/- {cv.get('std_best_val_overall_mae', 0):.3f} cm")

            tr2 = Trainer(model=make_model(cfg), config=cfg, show_progress=True)
            tr2.fit_final(train_pool_s, val_frac=0.15)
            out = tr2.evaluate(test_s); m = out["metrics"]

            groups = {"beam_conn": [], "beam_iso": [], "col_conn": [], "col_iso": []}
            bt, bp, ct, cp = [], [], [], []; persample = []
            for g, pred in zip(test_s, out["predictions"]):
                name = getattr(g, "sample_name", "graph"); gerrs = []
                for nt, tt, pp, gc, gi in [("beam", bt, bp, "beam_conn", "beam_iso"),
                                           ("column", ct, cp, "col_conn", "col_iso")]:
                    if nt not in pred or not hasattr(g[nt], "y"):
                        continue
                    true = g[nt].y.cpu().numpy(); P = pred[nt]
                    tt.append(true); pp.append(P)
                    err = np.abs(P - true).mean(axis=1); gerrs.append(err)
                    wasiso = getattr(g[nt], "was_isolated", None)
                    wi = (wasiso.cpu().numpy().astype(bool)
                          if wasiso is not None else np.zeros(true.shape[0], dtype=bool))
                    groups[gi].append(err[wi]); groups[gc].append(err[~wi])
                    for j in range(true.shape[0]):
                        pernode_rows.append({
                            "combo": tag, "sample": name, "node_type": nt, "node_idx": j,
                            "was_isolated": bool(wi[j]),
                            "true_b": float(true[j, 0]), "true_h": float(true[j, 1]),
                            "pred_b": float(P[j, 0]), "pred_h": float(P[j, 1]),
                            "abs_err_b": float(abs(P[j, 0] - true[j, 0])),
                            "abs_err_h": float(abs(P[j, 1] - true[j, 1])),
                            "node_mae": float(err[j])})
                if gerrs:
                    allg = np.concatenate(gerrs)
                    persample.append((name, int(allg.size), float(allg.mean())))

            BT_ = np.concatenate(bt) if bt else np.empty((0, 2))
            BP_ = np.concatenate(bp) if bp else np.empty((0, 2))
            CT_ = np.concatenate(ct) if ct else np.empty((0, 2))
            CP_ = np.concatenate(cp) if cp else np.empty((0, 2))
            sweep_preds[(pe_mode, iso)] = {"beam_true": BT_, "beam_pred": BP_,
                                           "col_true": CT_, "col_pred": CP_}
            beam_r, col_r = _reg(BT_, BP_, hit_tol), _reg(CT_, CP_, hit_tol)
            all_t = np.concatenate([x for x in (BT_, CT_) if x.shape[0]]) if (bt or ct) else np.empty((0, 2))
            all_p = np.concatenate([x for x in (BP_, CP_) if x.shape[0]]) if (bp or cp) else np.empty((0, 2))
            over_r = _reg(all_t, all_p, hit_tol)

            base_pred = {}
            for nt in ("beam", "column"):
                ys = [g[nt].y.cpu().numpy() for g in train_pool_s
                      if nt in g.node_types and hasattr(g[nt], "y") and g[nt].y is not None]
                if ys:
                    base_pred[nt] = np.median(np.concatenate(ys), axis=0)
            base_err, base_mae_type = [], {}
            for nt, TT in (("beam", BT_), ("column", CT_)):
                if nt in base_pred and TT.shape[0]:
                    e = np.abs(base_pred[nt] - TT).mean(axis=1)
                    base_mae_type[nt] = float(e.mean()); base_err.append(e)
            base_weighted = float(np.concatenate(base_err).mean()) if base_err else float("nan")

            def gstat(key):
                a = [x for x in groups[key] if len(x)]
                if not a: return (float("nan"), 0)
                v = np.concatenate(a); return (float(v.mean()), int(v.size))
            bd = {"combo": tag}
            for key in ["beam_conn", "beam_iso", "col_conn", "col_iso"]:
                mae_, n_ = gstat(key); bd[f"{key}_MAE"] = round(mae_, 3); bd[f"{key}_n"] = n_
            breakdown_rows.append(bd)

            persample.sort(key=lambda x: -x[2])
            for nm, nn, e in persample:
                persample_rows.append({"combo": tag, "sample": nm, "n_nodes": nn, "MAE": round(e, 3)})

            beam_mae, col_mae = _g(m, "test_beam_mae"), _g(m, "test_column_mae")
            unw = _g(m, "test_unweighted_mae")
            if np.isnan(unw):
                unw = (beam_mae + col_mae) / 2
            sweep_rows.append({
                "PE": pe_mode, "isolated": iso, "k": (KNN_K if iso == "knn" else "-"),
                "weighted_MAE": round(over_r["mae"], 3), "unweighted_MAE": round(unw, 3),
                "beam_MAE": round(beam_mae, 3), "col_MAE": round(col_mae, 3),
                "width_MAE": round(over_r["width_mae"], 3), "height_MAE": round(over_r["height_mae"], 3),
                "beam_width_MAE": round(beam_r["width_mae"], 3), "beam_height_MAE": round(beam_r["height_mae"], 3),
                "col_width_MAE": round(col_r["width_mae"], 3), "col_height_MAE": round(col_r["height_mae"], 3),
                "pct_within_tol": round(100 * over_r["within_tol"], 1),
                "beam_within_tol": round(100 * beam_r["within_tol"], 1),
                "col_within_tol": round(100 * col_r["within_tol"], 1),
                "width_within_tol": round(100 * over_r["width_within_tol"], 1),
                "height_within_tol": round(100 * over_r["height_within_tol"], 1),
                "overall_RMSE": round(over_r["rmse"], 3),
                "beam_RMSE": round(beam_r["rmse"], 3), "col_RMSE": round(col_r["rmse"], 3),
                "overall_R2": round(over_r["r2"], 3), "width_R2": round(over_r["width_r2"], 3),
                "height_R2": round(over_r["height_r2"], 3),
                "beam_R2": round(beam_r["r2"], 3), "col_R2": round(col_r["r2"], 3),
                "baseline_MAE": round(base_weighted, 3),
                "baseline_beam_MAE": round(base_mae_type.get("beam", float("nan")), 3),
                "baseline_col_MAE": round(base_mae_type.get("column", float("nan")), 3),
                "improve_vs_baseline": round(base_weighted - over_r["mae"], 3),
                "cv_MAE": round(cv.get("mean_best_val_overall_mae", float("nan")), 3),
                "cv_MAE_std": round(cv.get("std_best_val_overall_mae", float("nan")), 3),
                "model_dir": f"{models_dir}/{tag}", "status": "ok"})
            print(f"\n--> COMBO {ci} DONE: weighted {sweep_rows[-1]['weighted_MAE']} | "
                  f"beam {sweep_rows[-1]['beam_MAE']} | col {sweep_rows[-1]['col_MAE']} cm | "
                  f"<= {hit_tol:.0f}cm {sweep_rows[-1]['pct_within_tol']}%\n")
        except Exception as e:
            import traceback
            print(f"\n!! COMBO {ci} FAILED: {e}\n"); traceback.print_exc()
            sweep_rows.append({"PE": pe_mode, "isolated": iso, "status": f"ERROR: {e}",
                               "weighted_MAE": float("nan"), "model_dir": "-"})

    results_table = pd.DataFrame(sweep_rows).sort_values("weighted_MAE").reset_index(drop=True)
    results_table.to_csv(f"{results_dir}/pe_isolated_sweep.csv", index=False)
    pd.DataFrame(breakdown_rows).to_csv(f"{results_dir}/pe_isolated_error_breakdown.csv", index=False)
    pd.DataFrame(persample_rows).to_csv(f"{results_dir}/pe_isolated_persample_errors.csv", index=False)
    pd.DataFrame(pernode_rows).to_csv(f"{results_dir}/pe_isolated_pernode_errors.csv", index=False)

    ok = results_table[results_table["status"] == "ok"]
    best_combo = None
    if len(ok):
        best = ok.iloc[0]; best_combo = (best["PE"], best["isolated"])
        with open(f"{results_dir}/best_model_summary.json", "w") as f:
            json.dump({"best": best.to_dict(), "selection_metric": "weighted_MAE",
                       "all_combos": ok.to_dict(orient="records")}, f, indent=2, default=float)
        print("#" * 72)
        print(f"BEST ({config['model']['type'].upper()}): PE={best['PE']} | iso={best['isolated']}"
              f" -> weighted {best['weighted_MAE']} cm")
        print("#" * 72)
    print(results_table.to_string(index=False))
    return results_table, best_combo


def deploy_and_test(config, best_combo, raw_train_filt, raw_test_filt,
                    results_dir, models_dir, hit_tol=HIT_TOL_CM):
    """Retrain the winning combo on ALL filtered train graphs, then evaluate on
    the (filtered) external test set. Saves results/filtered/<model>/test_*."""
    if best_combo is None:
        print("No successful combo -> skipping deployment/test."); return None
    pe_mode, iso = best_combo
    tag = f"{pe_mode}_{iso}_FULL"

    # deployment model on the whole filtered train set
    g_all = [g.clone() for g in raw_train_filt]
    IsolatedNodeHandler(strategy=iso, k=KNN_K).transform(g_all)
    PositionalEncoder(mode=pe_mode, dim=PE_DIM).transform(g_all)
    cfg = copy.deepcopy(config); cfg["paths"] = {"checkpoints": f"{models_dir}/{tag}"}
    tr = Trainer(model=make_model(cfg), config=cfg, show_progress=True)
    tr.fit_final(g_all, val_frac=0.15)
    print(f"Deployment model saved -> {models_dir}/{tag}/")

    # external test (already filtered) -> transform like training
    test_t = [g.clone() for g in raw_test_filt]
    IsolatedNodeHandler(strategy=iso, k=KNN_K).transform(test_t)
    PositionalEncoder(mode=pe_mode, dim=PE_DIM).transform(test_t)
    out = tr.evaluate(test_t)

    groups = {"beam_conn": [], "beam_iso": [], "col_conn": [], "col_iso": []}
    bt, bp, ct, cp = [], [], [], []; persample = []; pernode_rows = []
    for g, pred in zip(test_t, out["predictions"]):
        name = getattr(g, "sample_name", "graph"); gerrs = []
        for nt, tt, pp, gc, gi in [("beam", bt, bp, "beam_conn", "beam_iso"),
                                   ("column", ct, cp, "col_conn", "col_iso")]:
            if nt not in pred or not hasattr(g[nt], "y"):
                continue
            true = g[nt].y.cpu().numpy(); P = pred[nt]
            tt.append(true); pp.append(P)
            err = np.abs(P - true).mean(axis=1); gerrs.append(err)
            wasiso = getattr(g[nt], "was_isolated", None)
            wi = (wasiso.cpu().numpy().astype(bool) if wasiso is not None
                  else np.zeros(true.shape[0], dtype=bool))
            groups[gi].append(err[wi]); groups[gc].append(err[~wi])
            for j in range(true.shape[0]):
                pernode_rows.append({"sample": name, "node_type": nt, "node_idx": j,
                    "was_isolated": bool(wi[j]),
                    "true_b": float(true[j, 0]), "true_h": float(true[j, 1]),
                    "pred_b": float(P[j, 0]), "pred_h": float(P[j, 1]),
                    "abs_err_b": float(abs(P[j, 0] - true[j, 0])),
                    "abs_err_h": float(abs(P[j, 1] - true[j, 1])),
                    "node_mae": float(err[j])})
        if gerrs:
            a = np.concatenate(gerrs); persample.append((name, int(a.size), float(a.mean())))

    BT = np.concatenate(bt) if bt else np.empty((0, 2)); BP = np.concatenate(bp) if bp else np.empty((0, 2))
    CT = np.concatenate(ct) if ct else np.empty((0, 2)); CP = np.concatenate(cp) if cp else np.empty((0, 2))
    beam_r, col_r = _reg(BT, BP, hit_tol), _reg(CT, CP, hit_tol)
    allT = np.concatenate([x for x in (BT, CT) if x.shape[0]]) if (bt or ct) else np.empty((0, 2))
    allP = np.concatenate([x for x in (BP, CP) if x.shape[0]]) if (bp or cp) else np.empty((0, 2))
    over_r = _reg(allT, allP, hit_tol)

    base_pred = {}
    for nt in ("beam", "column"):
        ys = [g[nt].y.cpu().numpy() for g in raw_train_filt
              if nt in g.node_types and hasattr(g[nt], "y") and g[nt].y is not None]
        if ys: base_pred[nt] = np.median(np.concatenate(ys), axis=0)
    base_err = []
    for nt, TT in (("beam", BT), ("column", CT)):
        if nt in base_pred and TT.shape[0]:
            base_err.append(np.abs(base_pred[nt] - TT).mean(axis=1))
    base_w = float(np.concatenate(base_err).mean()) if base_err else float("nan")

    def gstat(k):
        a = [x for x in groups[k] if len(x)]
        if not a: return (float("nan"), 0)
        v = np.concatenate(a); return (float(v.mean()), int(v.size))

    tol = int(hit_tol)
    row = {
        "model": config["model"]["type"], "best_combo": f"{pe_mode}_{iso}",
        "weighted_MAE": round(over_r["mae"], 3),
        "unweighted_MAE": round((beam_r["mae"] + col_r["mae"]) / 2, 3),
        "beam_MAE": round(beam_r["mae"], 3), "col_MAE": round(col_r["mae"], 3),
        "width_MAE": round(over_r["width_mae"], 3), "height_MAE": round(over_r["height_mae"], 3),
        "overall_RMSE": round(over_r["rmse"], 3),
        "overall_R2": round(over_r["r2"], 3),
        "beam_R2": round(beam_r["r2"], 3), "col_R2": round(col_r["r2"], 3),
        f"pct_within_{tol}cm": round(100 * over_r["within_tol"], 1),
        "beam_within": round(100 * beam_r["within_tol"], 1),
        "col_within": round(100 * col_r["within_tol"], 1),
        "baseline_MAE": round(base_w, 3),
        "improve_vs_baseline": round(base_w - over_r["mae"], 3),
    }
    os.makedirs(results_dir, exist_ok=True)
    pd.DataFrame(pernode_rows).to_csv(f"{results_dir}/test_pernode_errors.csv", index=False)
    (pd.DataFrame([{"sample": n, "n_nodes": k, "MAE": round(e, 3)} for n, k, e in persample])
       .sort_values("MAE", ascending=False)
       .to_csv(f"{results_dir}/test_persample_errors.csv", index=False))
    bd = {}
    for k in ["beam_conn", "beam_iso", "col_conn", "col_iso"]:
        mae_, n_ = gstat(k); bd[f"{k}_MAE"] = round(mae_, 3); bd[f"{k}_n"] = n_
    pd.DataFrame([bd]).to_csv(f"{results_dir}/test_error_breakdown.csv", index=False)
    pd.DataFrame([row]).T.rename(columns={0: "value"}).to_csv(f"{results_dir}/test_metrics.csv")
    with open(f"{results_dir}/test_metrics.json", "w") as f:
        json.dump(row, f, indent=2, default=float)

    print(f"\n>>> [{config['model']['type'].upper()}] EXTERNAL TEST (filtered, {len(test_t)} graphs): "
          f"weighted MAE {row['weighted_MAE']} cm | within {tol}cm {row[f'pct_within_{tol}cm']}% "
          f"| beats baseline by {row['improve_vs_baseline']} cm")
    print(f"saved -> {results_dir}/test_metrics.json (+ csvs)")
    return row

print("helpers ready: run_sweep(), deploy_and_test()")


helpers ready: run_sweep(), deploy_and_test()


In [5]:
# ## Cell 5 — RUN: filter (train + test) -> sweep -> deploy -> external test, BOTH models
# WARNING: this trains 2 models x 9 combos x (5-fold CV + final) + 2 deployment
# models. It is heavy -> run on a GPU machine. Nothing here modifies the baseline.

# 1) filter ONCE (in-memory copies; raw_train/raw_test from Cell 3 stay intact)
filt = ImpossibleLabelFilter(max_impossible_cm=FILTER_MAX_CM, verbose=True)
train_filt = [g.clone() for g in raw_train]
filt.transform(train_filt)
print(f"TRAIN: removed {filt.n_removed} nodes {filt.n_removed_by_type}")

if APPLY_TO_TEST:
    filt_te = ImpossibleLabelFilter(max_impossible_cm=FILTER_MAX_CM, verbose=True)
    test_filt = [g.clone() for g in raw_test]
    filt_te.transform(test_filt)
    print(f"TEST : removed {filt_te.n_removed} nodes {filt_te.n_removed_by_type}")
else:
    test_filt = [g.clone() for g in raw_test]
    print("TEST : filter NOT applied (APPLY_TO_TEST=False)")

# 2) per model: sweep + deployment + external test
external_rows = []
for model_type in MODELS:
    print("\n" + "#" * 78)
    print(f"# MODEL = {model_type.upper()}   (filtered run)")
    print("#" * 78)
    cfg = load_model_config(model_type)
    results_dir = f"{RESULTS_ROOT}/{model_type}"
    models_dir  = f"{results_dir}/models"

    results_table, best_combo = run_sweep(cfg, train_filt, results_dir, models_dir)
    row = deploy_and_test(cfg, best_combo, train_filt, test_filt, results_dir, models_dir)
    if row is not None:
        external_rows.append(row)

print("\nDONE. Per-model outputs under", RESULTS_ROOT)


18:02:30 | ⚙️ SYSTEM | ℹ️  INFO     | ImpossibleLabelFilter removed 251 nodes (min(b,h) <= 10.0 cm): {'beam': 251, 'column': 0}
TRAIN: removed 251 nodes {'beam': 251, 'column': 0}
18:02:30 | ⚙️ SYSTEM | ℹ️  INFO     | ImpossibleLabelFilter removed 198 nodes (min(b,h) <= 10.0 cm): {'beam': 198, 'column': 0}
TEST : removed 198 nodes {'beam': 198, 'column': 0}

##############################################################################
# MODEL = HGT   (filtered run)
##############################################################################
[HGT] sweeping 9 combos | 5-fold CV + final each (~54 trainings).

COMBO 1/9 | PE=topological | isolated=none


18:02:30 | HGT      | INFO     | HGT created: 3 layers, 4 heads, 128 hidden
18:02:30 | TRAINER  | INFO     | Using CUDA GPU: NVIDIA GeForce RTX 4070 Laptop GPU
18:02:30 | TRAINER  | INFO     | Trainer ready on cuda | epochs=100, lr=0.001, folds=5, normalize=True
18:02:30 | TRAINER  | INFO     | 
5-Fold CV on 215 graphs (per-fold scaling)


18:02:31 | ⚙️ SYSTEM | ℹ️  INFO     | Fitted 'beam' scaler on 25606 nodes, 41 raw features
18:02:31 | ⚙️ SYSTEM | ℹ️  INFO     | Fitted 'column' scaler on 12577 nodes, 41 raw features
18:02:31 | ⚙️ SYSTEM | ℹ️  INFO     | Fitted 'beam' target scaler on 25606 nodes
18:02:31 | ⚙️ SYSTEM | ℹ️  INFO     | Fitted 'column' target scaler on 12577 nodes


18:02:31 | TRAINER  | INFO     | 
--- Fold 1/5: 172 train / 43 val ---
18:02:31 | HGT      | INFO     | Initializing HGT with metadata: (['beam', 'column'], [('beam', 'to', 'beam'), ('column', 'to', 'column'), ('beam', 'to', 'column'), ('column', 'to', 'beam')])
18:02:31 | HGT      | INFO     | HGT submodules initialized


fold_0 epochs:   0%|          | 0/100 [00:00<?, ?it/s]

...

final epochs:   0%|          | 0/100 [00:00<?, ?it/s]

02:23:58 | TRAINER  | INFO     | [final] ep 1/100 | train 0.7070 | val 0.6525 | beam MAE 6.183 R2 0.425 | col MAE 7.494 R2 0.023
02:24:39 | TRAINER  | INFO     | [final] ep 10/100 | train 0.4005 | val 0.5290 | beam MAE 5.382 R2 0.548 | col MAE 6.246 R2 0.235
02:25:26 | TRAINER  | INFO     | [final] ep 20/100 | train 0.3398 | val 0.4783 | beam MAE 5.125 R2 0.588 | col MAE 5.598 R2 0.322
02:26:14 | TRAINER  | INFO     | [final] ep 30/100 | train 0.3070 | val 0.5230 | beam MAE 5.310 R2 0.564 | col MAE 5.848 R2 0.326
02:27:01 | TRAINER  | INFO     | [final] ep 40/100 | train 0.2694 | val 0.5494 | beam MAE 5.173 R2 0.571 | col MAE 6.103 R2 0.301
02:27:11 | TRAINER  | INFO     | [final] early stop at epoch 42


02:27:11 | ⚙️ SYSTEM | ℹ️  INFO     | Normalizer saved to ..\results\filtered\han\models\hybrid_self_loop_FULL\feature_normalizer.pkl
02:27:11 | ⚙️ SYSTEM | ℹ️  INFO     | Target normalizer saved to ..\results\filtered\han\models\hybrid_self_loop_FULL\target_normalizer.pkl
Deployment model saved -> ../results/filtered/han/models/hybrid_self_loop_FULL/


02:27:11 | TRAINER  | INFO     | 
Evaluating on 59 graphs
02:27:12 | TRAINER  | INFO     | 
Test results (real units):
02:27:12 | TRAINER  | INFO     |   test_beam_mae               : 5.1220
02:27:12 | TRAINER  | INFO     |   test_beam_mse               : 45.8479
02:27:12 | TRAINER  | INFO     |   test_beam_rmse              : 6.7711
02:27:12 | TRAINER  | INFO     |   test_beam_width_mae         : 5.1390
02:27:12 | TRAINER  | INFO     |   test_beam_height_mae        : 5.1050
02:27:12 | TRAINER  | INFO     |   test_beam_r2                : 0.5183
02:27:12 | TRAINER  | INFO     |   test_column_mae             : 3.8165
02:27:12 | TRAINER  | INFO     |   test_column_mse             : 29.3443
02:27:12 | TRAINER  | INFO     |   test_column_rmse            : 5.4170
02:27:12 | TRAINER  | INFO     |   test_column_width_mae       : 4.2198
02:27:12 | TRAINER  | INFO     |   test_column_height_mae      : 3.4132
02:27:12 | TRAINER  | INFO     |   test_column_r2              : 0.4877
02:27:12 | TRAI


>>> [HAN] EXTERNAL TEST (filtered, 59 graphs): weighted MAE 4.701 cm | within 5cm 63.2% | beats baseline by 2.186 cm
saved -> ../results/filtered/han/test_metrics.json (+ csvs)

DONE. Per-model outputs under ../results/filtered


In [6]:
# ## Cell 6 — Compare filtered HGT vs HAN, and each vs its (unfiltered) baseline
rows = []
for model_type in MODELS:
    filt_json = f"{RESULTS_ROOT}/{model_type}/test_metrics.json"
    base_json = f"../results/{model_type}/test_metrics.json"   # baseline (untouched)
    if not os.path.exists(filt_json):
        print(f"[skip] {filt_json} not found (did Cell 5 finish for {model_type}?)"); continue
    with open(filt_json) as f:
        fm = json.load(f)
    bm = {}
    if os.path.exists(base_json):
        with open(base_json) as f:
            bm = json.load(f)
    def pick(d, *keys):
        for k in keys:
            if k in d: return d[k]
        return float("nan")
    rows.append({
        "model": model_type,
        "baseline_weighted_MAE": pick(bm, "weighted_MAE"),
        "filtered_weighted_MAE": pick(fm, "weighted_MAE"),
        "baseline_within5cm": pick(bm, "pct_within_5cm"),
        "filtered_within5cm": pick(fm, "pct_within_5cm"),
        "baseline_beam_MAE": pick(bm, "beam_MAE"),
        "filtered_beam_MAE": pick(fm, "beam_MAE"),
        "baseline_col_MAE": pick(bm, "col_MAE"),
        "filtered_col_MAE": pick(fm, "col_MAE"),
        "filtered_best_combo": pick(fm, "best_combo"),
    })

cmp = pd.DataFrame(rows)
if len(cmp):
    cmp["MAE_delta(filt-base)"] = cmp["filtered_weighted_MAE"] - cmp["baseline_weighted_MAE"]
    os.makedirs(RESULTS_ROOT, exist_ok=True)
    cmp.to_csv(f"{RESULTS_ROOT}/comparison_filtered_vs_baseline.csv", index=False)
    print("Filtered vs baseline (external test, weighted MAE in cm):")
    print(cmp.to_string(index=False))
    print(f"\nsaved -> {RESULTS_ROOT}/comparison_filtered_vs_baseline.csv")
    print("\nNote: baseline test MAE is measured against the RAW test labels "
          "(incl. impossible ones), while the filtered MAE is measured against the "
          "cleaned test labels, so the delta reflects BOTH the cleaner training "
          "signal AND the cleaner evaluation target.")
else:
    print("No filtered results found yet - run Cell 5 first.")


Filtered vs baseline (external test, weighted MAE in cm):
model  baseline_weighted_MAE  filtered_weighted_MAE  baseline_within5cm  filtered_within5cm  baseline_beam_MAE  filtered_beam_MAE  baseline_col_MAE  filtered_col_MAE filtered_best_combo  MAE_delta(filt-base)
  hgt                 4.9300                 4.5940             62.8000             63.6000             5.4610             4.9640            3.7930            3.8160          hybrid_knn               -0.3360
  han                 5.0370                 4.7010             62.0000             63.2000             5.5730             5.1220            3.8880            3.8160    hybrid_self_loop               -0.3360

saved -> ../results/filtered/comparison_filtered_vs_baseline.csv

Note: baseline test MAE is measured against the RAW test labels (incl. impossible ones), while the filtered MAE is measured against the cleaned test labels, so the delta reflects BOTH the cleaner training signal AND the cleaner evaluation target.


## Notes — how to read this, and what stays fixed

**Metrics** are identical to notebooks 5 & 6 (all in cm except R²), computed on
inverse-transformed predictions:

- `weighted_MAE` — every node counts equally (selection metric).
- `unweighted_MAE` — mean of beam / column MAE (each type counts equally).
- `pct_within_5cm` — share of b/h predictions with `|pred − true| ≤ 5 cm`.
- `baseline_MAE` — per-type median (b,h) of the **filtered** training targets.
- `cv_MAE ± cv_MAE_std` — k-fold generalization estimate on the 85% pool.

**Outputs** (mirror the baseline layout, under a separate root):

```
results/filtered/hgt/pe_isolated_sweep.csv, ...pernode/persample/breakdown.csv,
                     best_model_summary.json, models/<combo>/...,
                     test_metrics.json/csv, test_*_errors.csv
results/filtered/han/ (same)
results/filtered/comparison_filtered_vs_baseline.csv
```

**What is intentionally NOT changed** (so the comparison stays valid):

- `data/graphs/*.pt` and every Excel file — read-only.
- Notebooks 3, 5, 6 and everything in `results/hgt`, `results/han`, `results/analysis`.
- The model architectures, the trainer, the PE / isolated-node transforms.

The only new code is `ImpossibleLabelFilter` in `src/data_manager/data_processor.py`
(opt-in; nothing else imports it) and this notebook. Turn the cleaning off by
setting `MODELS`/`APPLY_TO_TEST` or, at the transform level, by never calling the
filter — the baseline path is unaffected either way.
